# Семинар: Статистические тесты в Machine Learning

## Цель семинара

Научиться применять статистические методы для оценки надежности метрик ML-моделей, сравнения моделей между собой, отбора признаков (Feature Selection), оценки качества классификации, детекции дрейфа данных (Data Drift).

## План семинара

1. Доверительные интервалы и проверка гипотез — как понять, что метрика модели надежна
2. Тест Фишера и хи-квадрат — анализ матрицы ошибок и отбор категориальных признаков
3. KS-тест — оценка качества классификации и детекция Data Drift

---

# Блок 1. Доверительные интервалы и проверка гипотез

## Проблема точечных оценок

Представим, что мы обучили модель классификации спама. На тестовой выборке из $n=1000$ писем она ошиблась в 150 случаях. Accuracy получаем: $1 - 150/1000 = 0.85$. Мы идём к заказчику и говорим: "Точность нашей модели — 85%".

С точки зрения строгой статистики, мы только что солгали.


<details>
<summary><b>Почему?</b></summary>

Потому что 0.85 — это точечная оценка ($\bar{X}$). Она верна только для этой конкретной тысячи писем. Завтра придет новая тысяча, и точность может быть 0.82 или 0.87.
</details>

## Центральная Предельная Теорема (ЦПТ)

Формула из лекций:

$$\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \sim \mathcal{N}(0,1)$$

Это Центральная Предельная Теорема (ЦПТ). С её помощью мы можем сказать, что если мы будем много раз брать новые тестовые выборки, распределение наших оценок Accuracy будет стремиться к колокообразной кривой (нормальному распределению) вокруг истинной, но неизвестной точности модели ($\mu$).

## Таблица соответствия: Статистика ↔ ML

| Символ из формулы | Классическая статистика | Машинное обучение (интерпретация) |
| :--- | :--- | :--- |
| $\bar{X}$ | Выборочное среднее | Наша метрика на тесте (например, Accuracy = 0.85) |
| $\mu$ | Истинное математическое ожидание | Истинная метрика модели на *всех возможных данных* |
| $\sigma$ | Стандартное отклонение | Разброс предсказаний. Для бинарной классификации $\sigma = \sqrt{p(1-p)}$ |
| $\frac{\sigma}{\sqrt{n}}$ | Стандартная ошибка (Standard Error) | Неопределенность нашей оценки. Чем больше $n$, тем меньше эта величина |
| $\Phi_{\mathcal{N}(0,1)}(k)$ | Функция нормального распределения | Позволяет сказать: "С вероятностью 95% истинное $\mu$ лежит вот в этом коридоре" |

## Логика проверки гипотезы

<details>
<summary><b>Какова наша нулевая гипотеза?</b></summary>

Мы хотим проверить нулевую гипотезу $H_0: p = 0.5$ ("Модель не лучше случайного угадывания") против двусторонней альтернативы $H_1: p \neq 0.5$.

Для проверки мы используем двухвыборочный Z-критерий для долей (двусторонний тест на уровне значимости $\alpha = 0.05$). Если наблюдаемая нами оценка точности $\hat{p}$ (или $\bar{X}$) значимо отклоняется от гипотетического значения $0.5$ (то есть попадает в критическую область критерия), мы **отвергаем** нулевую гипотезу $H_0$. 

*Примечание: Проверка этой гипотезы математически эквивалентна проверке того, попадает ли значение $0.5$ в инвертированный доверительный интервал, построенный на основе дисперсии нулевой гипотезы.*
</details>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. Настройка параметров "эксперимента"
np.random.seed(123)
n = 500               # Размер тестовой выборки
true_accuracy = 0.53  # Истинная точность модели (параметр p)
null_hypothesis_p = 0.50 # H0: Модель работает на уровне случайного угадывания (p0 = 0.5)

# 2. Симуляция для демонстрации ЦПТ (1000 выборок)
num_simulations = 1000
sample_accuracies = np.mean(np.random.binomial(1, true_accuracy, size=(num_simulations, n)), axis=1)

# 3. Наблюдаемые данные (одна конкретная выборка ML-инженера)
observed_accuracy = np.mean(np.random.binomial(1, true_accuracy, n))

# --- А. РАСЧЕТ ДОВЕРИТЕЛЬНОГО ИНТЕРВАЛА (Используем выборочную дисперсию) ---
sigma_sample = np.sqrt(observed_accuracy * (1 - observed_accuracy))
se_sample = sigma_sample / np.sqrt(n)
z_crit = stats.norm.ppf(0.975) 
ci_lower = observed_accuracy - z_crit * se_sample
ci_upper = observed_accuracy + z_crit * se_sample

# --- Б. ПРОВЕРКА ГИПОТЕЗЫ Z-ТЕСТОМ (Используем нулевую дисперсию из H0) ---
se_null = np.sqrt(null_hypothesis_p * (1 - null_hypothesis_p) / n)
z_stat = (observed_accuracy - null_hypothesis_p) / se_null
# Двустороннее p-value
p_value = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))

# 4. ВИЗУАЛИЗАЦИЯ
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(sample_accuracies, bins=40, kde=True, stat="density", color="skyblue", 
             label="Распределение оценок Accuracy (моделирование ЦПТ)", ax=ax)

# Рассчитываем критические границы метрики на оси X для гипотезы H0
p_crit_lower = null_hypothesis_p - z_crit * se_null
p_crit_upper = null_hypothesis_p + z_crit * se_null

# Визуализируем зону неотвержения H0
ax.axvspan(p_crit_lower, p_crit_upper, color='red', alpha=0.1, 
           label=f'Зона неотвержения H₀ (α=0.05): [{p_crit_lower:.3f}, {p_crit_upper:.3f}]')
ax.axvline(p_crit_lower, color='red', linestyle=':', linewidth=1.5)
ax.axvline(p_crit_upper, color='red', linestyle=':', linewidth=1.5)

# Линия Нулевой Гипотезы
ax.axvline(null_hypothesis_p, color='red', linestyle='-.', linewidth=3, label=f'Нулевая гипотеза ($H_0: p_0 = {null_hypothesis_p}$)')

# Аннотация на основе статистического критерия (уровень значимости alpha = 0.05)
alpha = 0.05
if p_value < alpha:
    result_text = f"H0 ОТВЕРГАЕТСЯ\np-value = {p_value:.4e}\nМодель значимо лучше случайности"
    text_color = 'darkgreen'
else:
    result_text = f"H0 НЕ ОТВЕРГАЕТСЯ\np-value = {p_value:.4f}\nРазница может быть случайным шумом"
    text_color = 'darkred'

ax.text(0.98, 0.85, result_text, transform=ax.transAxes, fontsize=12, fontweight='bold', 
        color=text_color, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=text_color, linewidth=2))

ax.set_title("Проверка гипотез и ДИ для биномиальных долей (Accuracy)", fontsize=15, fontweight='bold')
ax.set_xlabel("Значение метрики (Accuracy)", fontsize=12)
ax.set_ylabel("Плотность вероятности", fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Наблюдаемая Accuracy: {observed_accuracy:.4f}")
print(f"95% Доверительный интервал Вальда: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Статистика Z-критерия: {z_stat:.4f}")
print(f"p-value: {p_value:.4e}")

## Что мы увидели на графике

1. Голубой "холм" (гистограмма) — это графическая визуализация Центральной Предельной Теоремы. Даже если предсказания модели на отдельных объектах просто нули и единицы (бернуллиевские испытания в нашем случае), *выборочное среднее* этих предсказаний (Accuracy) при многократном тестировании образует красивый нормальный «колокол».
2. Красный прозрачный коридор — это графическое представление зоны неотвержения нулевой гипотезы $H_0$ (границы: $[0.456, 0.544]$). Он очерчивает рамки, в которые с вероятностью 95% попала бы метрика Accuracy, если бы наша модель на самом деле работала **не лучше обычного случайного угадывания (50%)**.
3. Вертикальная красная штрихпунктирная линия ($H_0$) — центр случайного распределения ($p_0 = 0.5$). Бизнес часто задает вопрос: *«А вдруг модель просто угадывает ответы как монетка?»*. Мы видим, что истина находится левее основного холма нашей модели.
4. Результат проверки гипотезы ($H_0$ ОТВЕРГАЕТСЯ) — наблюдаемая оценка точности нашей модели находится на отметке около **0.545**. Она едва-едва перешагнула тонкую красную пунктирную линию — правую границу случайной зоны. 
   
   Поскольку метрика вышла за пределы красного коридора, формально мы **отвергаем нулевую гипотезу**. Полученное значение $p$-value равно **$0.0318$** (или $3.18 \times 10^{-2}$), что меньше стандартного порога значимости $\alpha = 0.05$. 

> ⚠️ **Вывод ML-инженера:** Модель статистически значимо лучше случайности, но она балансирует на самом краю критической зоны. Эффект очень слабый ($p$-value близко к $0.05$), поэтому в реальном продакшене полагаться на такую модель крайне опасно — на новых данных она легко может провалиться обратно в зону случайного шума.


# Блок 2. Точный тест Фишера и критерий хи-квадрат

## Когда применять эти тесты?

В первом блоке мы работали с числами (метриками). Но что делать, если у нас мало данных? Или если мы хотим понять, зависит ли категориальный признак от целевой переменной?

В ML эти тесты решают две задачи:
1. Анализ матрицы ошибок на малых данных. Т.е. работает ли модель лучше случайности?
2. Отбор признаков (Feature Selection). Т.е. зависит ли этот категориальный признак от целевой переменной?

### Точный тест Фишера: Анализ матрицы ошибок на малых данных

У нас есть тестовая выборка из 20 объектов. Модель предсказала:
* 14 правильных предсказаний ($TP + TN$)
* 6 ошибок ($FP + FN$)

Вопрос: статистически ли значимо это преимущество над случайным угадыванием (50%)?

Мы строим таблицу сопряженности $2\times2$:

| | Предсказано: Класс 1 | Предсказано: Класс 0 | Сумма |
| :--- | :---: | :---: | :---: |
| **Истинно: Класс 1** | 10 (TP) | 2 (FN) | **12** |
| **Истинно: Класс 0** | 4 (FP) | 4 (TN) | **8** |
| **Сумма** | **14** | **6** | **20** |


Точный тест Фишера отвечает на вопрос: "Если бы модель угадывала случайно (распределяя 14 правильных ответов и 6 ошибок случайным образом при фиксированных суммах строк и столбцов), какова вероятность получить такую или еще более экстремальную таблицу?"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact

# Матрица ошибок (пример из нарратива)
# [[True Negative, False Positive], [False Negative, True Positive]]
confusion_matrix = np.array([[4, 4],  # Истинно Класс 0
                             [2, 10]])  # Истинно Класс 1

# Точный тест Фишера
odds_ratio, p_value = fisher_exact(confusion_matrix)

# === ИСПРАВЛЕННАЯ ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: Heatmap матрицы ошибок
# УБРАЛИ annot=True, чтобы не было дублирования
sns.heatmap(confusion_matrix, fmt='d', cmap='Blues',
            cbar=False, ax=axes[0], linewidths=2, linecolor='black',
            xticklabels=['Предсказано: 0', 'Предсказано: 1'],
            yticklabels=['Истинно: 0', 'Истинно: 1'])
axes[0].set_title('Матрица ошибок (Confusion Matrix)\nN = 20 объектов',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Предсказание модели', fontsize=11)
axes[0].set_ylabel('Истинный класс', fontsize=11)

# Добавим подписи ячеек ВРУЧНУЮ с правильной логикой цвета
# Порог для выбора цвета: если значение > 50% от максимума — белый текст, иначе чёрный
threshold = confusion_matrix.max() * 0.5

for i in range(2):
    for j in range(2):
        value = confusion_matrix[i, j]
        # Выбираем цвет текста в зависимости от яркости ячейки
        text_color = 'white' if value > threshold else 'black'

        axes[0].text(j + 0.5, i + 0.5,
                     f'{value}',
                     ha='center', va='center',
                     fontsize=18, fontweight='bold',
                     color=text_color)

# Правый график: Интерпретация p-value
colors = ['red' if p_value > 0.05 else 'green']
bars = axes[1].bar(['p-value'], [p_value], color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=2, label='Порог значимости α = 0.05')
axes[1].set_ylim(0, max(p_value * 1.5, 0.1))
axes[1].set_ylabel('p-value', fontsize=12)
axes[1].set_title('Результат точного теста Фишера', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# Добавим текстовую аннотацию
if p_value > 0.05:
    conclusion = "H₀ НЕ ОТВЕРГАЕТСЯ\nМодель работает не лучше случайности"
    axes[1].text(0.5, p_value * 0.7, conclusion, ha='center', va='center',
                 fontsize=12, fontweight='bold', color='red',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='red', linewidth=2))
else:
    conclusion = "H₀ ОТВЕРГАЕТСЯ\nМодель статистически значимо лучше случайности"
    axes[1].text(0.5, p_value * 0.7, conclusion, ha='center', va='center',
                 fontsize=12, fontweight='bold', color='green',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='green', linewidth=2))

# Добавим информацию о odds ratio
axes[1].text(0.5, 0.02, f'Odds Ratio = {odds_ratio:.2f}',
             ha='center', va='bottom', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', edgecolor='blue'))

plt.tight_layout()
plt.show()

print(f"Odds Ratio: {odds_ratio:.3f}")
print(f"p-value: {p_value:.4f}")
print(
    f"Вывод: {'Модель работает НЕ лучше случайности' if p_value > 0.05 else 'Модель статистически значимо лучше случайности'}")

## Критерий хи-квадрат: Отбор категориальных признаков

Теперь представим, что у нас 1000 объектов и 50 категориальных признаков (например, "Город", "Профессия", "Тип жилья"). Мы хотите понять, какие из этих признаков статистически значимо связаны с целевой переменной (например, "Купит ли клиент продукт?").

Для каждого признака мы строим таблицу сопряженности и применяем тест хи-квадрат. Если p-value < 0.05 — признак значим, оставляем его. Если p-value > 0.05 — признак не связан с целевой переменной, выбрасываем.

Это классический метод отбора признаков (Feature Selection).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Пример: зависимость "Города" от "Покупки продукта"
contingency_table = np.array([
    [80, 120],   # Москва: 120 не купили, 80 купили
    [90, 110],   # СПб: 90 не купили, 110 купили
    [50, 150]    # Казань: 150 не купили, 50 купили
])

# Хи-квадрат тест
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

# ВИЗУАЛИЗАЦИЯ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: Наблюдаемые vs Ожидаемые частоты
x = np.arange(len(contingency_table))
width = 0.35

observed_0 = contingency_table[:, 0]  # Не купили
observed_1 = contingency_table[:, 1]  # Купили

expected_0 = expected[:, 0]
expected_1 = expected[:, 1]

bars1 = axes[0].bar(x - width/2, observed_0, width, label='Наблюдаемые (Не купили)',
                    color='lightcoral', edgecolor='black', linewidth=1.5)
bars2 = axes[0].bar(x + width/2, observed_1, width, label='Наблюдаемые (Купили)',
                    color='lightgreen', edgecolor='black', linewidth=1.5)

# Добавим ожидаемые частоты как точки
axes[0].scatter(x - width/2, expected_0, color='darkred', marker='D', s=100,
               label='Ожидаемые (Не купили)', zorder=5)
axes[0].scatter(x + width/2, expected_1, color='darkgreen', marker='D', s=100,
               label='Ожидаемые (Купили)', zorder=5)

axes[0].set_xlabel('Город', fontsize=12)
axes[0].set_ylabel('Количество клиентов', fontsize=12)
axes[0].set_title('Наблюдаемые vs Ожидаемые частоты\n(Если бы города не влияли на покупку)',
                  fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Москва', 'СПб', 'Казань'])
axes[0].legend(fontsize=9, loc='upper right')
axes[0].grid(axis='y', alpha=0.3)

# Правый график: p-value
color = 'red' if p_value > 0.05 else 'green'
axes[1].bar(['p-value'], [p_value], color=color, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[1].set_ylim(0, max(p_value * 1.5, 0.001))
axes[1].set_ylabel('p-value', fontsize=12)
axes[1].set_title(f'Хи-квадрат тест\nχ² = {chi2_stat:.2f}, df = {dof}',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

if p_value > 0.05:
    conclusion = "H₀ НЕ ОТВЕРГАЕТСЯ\nГород НЕ влияет на покупку\n→ Признак можно выбросить"
    axes[1].text(0.5, p_value * 0.6, conclusion, ha='center', va='center',
                fontsize=11, fontweight='bold', color='red',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='red', linewidth=2))
else:
    conclusion = "H₀ ОТВЕРГАЕТСЯ\nГород ВЛИЯЕТ на покупку\n→ Признак оставляем"
    axes[1].text(0.5, p_value * 0.6, conclusion, ha='center', va='center',
                fontsize=11, fontweight='bold', color='green',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='green', linewidth=2))

plt.tight_layout()
plt.show()

print(f"Хи-квадрат статистика: {chi2_stat:.3f}")
print(f"Степени свободы: {dof}")
print(f"p-value: {p_value:.6f}")
print(f"Вывод: {'Признак НЕ значим' if p_value > 0.05 else 'Признак статистически значим'}")

## Когда какой тест применять?

| Критерий | Когда применять | Размер выборки | ML-задача |
| :--- | :--- | :--- | :--- |
| **Точный тест Фишера** | Малые выборки, разреженные таблицы | N < 30-40 | Анализ матрицы ошибок на маленьком тесте |
| **Хи-квадрат** | Большие выборки | N > 30-40 | Отбор категориальных признаков (Feature Selection) |

---

# Блок 3. Критерий Колмогорова-Смирнова (KS-тест)

## Оценка качества бинарной классификации

Мы обучили модель, которая предсказывает вероятность дефолта клиента. У нас есть тестовая выборка с истинными метками. Мы хотим ответить на вопрос: насколько хорошо модель разделяет классы?

- Если KS-статистика близка к **1** — распределения вероятностей для классов 0 и 1 полностью разделены (идеальная модель)
- Если KS-статистика близка к **0** — распределения сливаются (модель не лучше случайного угадывания)

На практике KS-статистика тесно связана с **ROC-AUC**: чем выше KS, тем выше AUC. Но KS даёт нам визуальную интерпретацию — мы можем наглядно увидеть, в какой точке предсказаний модели разделяют классы лучше всего.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from statsmodels.distributions.empirical_distribution import ECDF

# 1. Генерируем данные с хорошей разделяемостью
np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=10, class_sep=1.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Обучаем модель и получаем предсказанные вероятности
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 3. Разделяем предсказания по классам
y_pred_pos = y_pred_proba[y_test == 1]  # Класс 1 (например, Дефолт)
y_pred_neg = y_pred_proba[y_test == 0]  # Класс 0 (Не дефолт)

# 4. Считаем KS-статистику через scipy
ks_stat, p_value = stats.ks_2samp(y_pred_pos, y_pred_neg)

# 5. Строим строгие эмпирические CDF с помощью statsmodels
ecdf_pos = ECDF(y_pred_pos)
ecdf_neg = ECDF(y_pred_neg)

# 6. Находим точную точку максимального расхождения
# Оцениваем обе CDF на сетке из всех уникальных предсказанных точек
all_points = np.unique(np.concatenate([y_pred_pos, y_pred_neg]))
cdf_pos_vals = ecdf_pos(all_points)
cdf_neg_vals = ecdf_neg(all_points)

diffs = np.abs(cdf_pos_vals - cdf_neg_vals)
max_idx = np.argmax(diffs)
ks_point = all_points[max_idx]

# Значения CDF в точке максимального расхождения для графика
cdf_pos_at_ks = ecdf_pos(ks_point)
cdf_neg_at_ks = ecdf_neg(ks_point)

# ВИЗУАЛИЗАЦИЯ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: Распределения (гистограммы)
axes[0].hist(y_pred_pos, bins=30, alpha=0.6, color='green', label='Class 1 (Default)', density=True)
axes[0].hist(y_pred_neg, bins=30, alpha=0.6, color='red', label='Class 0 (No Default)', density=True)
axes[0].set_xlabel('Предсказанная вероятность дефолта', fontsize=11)
axes[0].set_ylabel('Плотность', fontsize=11)
axes[0].set_title('Распределения предсказанных вероятностей', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Правый график: CDF с явным показом KS-статистики
# Использование step(..., x, y) из ECDF: x содержит граничные точки, y — вероятности
axes[1].step(ecdf_neg.x, ecdf_neg.y, where='post', color='red', linewidth=2.5, label='CDF Class 0')
axes[1].step(ecdf_pos.x, ecdf_pos.y, where='post', color='green', linewidth=2.5, label='CDF Class 1')

# Вертикальная линия в точке максимального расхождения
axes[1].axvline(ks_point, color='black', linestyle='--', linewidth=1.5, alpha=0.5)

# Горизонтальный маркер KS-статистики (синяя линия)
axes[1].plot([ks_point, ks_point], [cdf_neg_at_ks, cdf_pos_at_ks],
             color='blue', linewidth=4, solid_capstyle='round',
             label=f'KS-статистика = {ks_stat:.3f}')

# Точки на концах отрезка
axes[1].scatter([ks_point, ks_point], [cdf_neg_at_ks, cdf_pos_at_ks],
                color='blue', s=100, zorder=5, edgecolor='black', linewidth=1.5)

axes[1].set_xlabel('Предсказанная вероятность дефолта', fontsize=11)
axes[1].set_ylabel('CDF (кумулятивная вероятность)', fontsize=11)
axes[1].set_title(f'KS-тест: Максимальное расстояние между CDF\np-value = {p_value:.2e}',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(alpha=0.3)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"KS-статистика: {ks_stat:.3f}")
print(f"p-value: {p_value:.2e}")
print(f"Точка максимального расхождения: {ks_point:.3f}")
print(f"Интерпретация: {'Модель хорошо разделяет классы' if ks_stat > 0.5 else 'Модель плохо разделяет классы'}")

## Детекция дрейфа данных (Data Drift Detection)

Мы внедрили модель в продакшен. Прошло 3 месяца. Как понять, что модель всё ещё работает корректно?

Представим, что в обучающей выборке средний доход клиента был 80 000 рублей. Но в продакшене пришли клиенты с другим распределением доходов (например, из нового региона).

Когда распределение признака статистически значимо изменилось, это и называют **Data Drift**. Модель, обученная на старых данных, может работать некорректно в новых условиях, когда данные сместились.

KS-тест идеально подходит для этой задачи, поскольку он может сравнить распределение признака в train и production. Если p-value < 0.05, то распределения разные, нужен мониторинг данных или переобучение модели.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 1. Генерируем "обучающую" выборку (train)
np.random.seed(12)
train_income = np.random.normal(loc=80000, scale=20000, size=1000)

# 2. Генерируем "продакшен" выборку с дрейфом (production)
# Сдвиг среднего на 15000 и увеличение дисперсии
prod_income = np.random.normal(loc=95000, scale=25000, size=1000)

# 3. KS-тест
ks_stat, p_value = stats.ks_2samp(train_income, prod_income)

# 4. Строим CDF
x_train = np.sort(train_income)
y_train = np.arange(1, len(x_train) + 1) / len(x_train)

x_prod = np.sort(prod_income)
y_prod = np.arange(1, len(x_prod) + 1) / len(x_prod)

# 5. Находим точку максимального расхождения
all_points = np.sort(np.concatenate([train_income, prod_income]))
cdf_train = np.searchsorted(x_train, all_points, side='right') / len(x_train)
cdf_prod = np.searchsorted(x_prod, all_points, side='right') / len(x_prod)
diffs = np.abs(cdf_train - cdf_prod)
ks_point = all_points[np.argmax(diffs)]
ks_max_diff = diffs.max()

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: Гистограммы распределений
axes[0].hist(train_income, bins=40, alpha=0.6, color='blue', label='Train (обучающая)', density=True)
axes[0].hist(prod_income, bins=40, alpha=0.6, color='orange', label='Production (продакшен)', density=True)
axes[0].set_xlabel('Доход клиента (руб.)', fontsize=11)
axes[0].set_ylabel('Плотность', fontsize=11)
axes[0].set_title('Распределение признака "Доход"', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Правый график: CDF с KS-статистикой
axes[1].step(x_train, y_train, where='post', color='blue', linewidth=2.5, label='CDF Train')
axes[1].step(x_prod, y_prod, where='post', color='orange', linewidth=2.5, label='CDF Production')

# Вертикальная линия в точке максимального расхождения
axes[1].axvline(ks_point, color='black', linestyle='--', linewidth=1.5, alpha=0.7)

# Горизонтальный отрезок KS-статистики
cdf_train_at_ks = np.searchsorted(x_train, ks_point, side='right') / len(x_train)
cdf_prod_at_ks = np.searchsorted(x_prod, ks_point, side='right') / len(x_prod)

axes[1].plot([ks_point, ks_point], [cdf_prod_at_ks, cdf_train_at_ks],
             color='red', linewidth=4, solid_capstyle='round',
             label=f'KS-статистика = {ks_stat:.3f}')

axes[1].scatter([ks_point, ks_point], [cdf_prod_at_ks, cdf_train_at_ks],
                color='red', s=100, zorder=5, edgecolor='black', linewidth=1.5)

axes[1].set_xlabel('Доход клиента (руб.)', fontsize=11)
axes[1].set_ylabel('CDF (кумулятивная вероятность)', fontsize=11)
axes[1].set_title(f'Data Drift Detection\np-value = {p_value:.2e}',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(alpha=0.3)

# Добавим аннотацию с выводом
if p_value < 0.05:
    conclusion = "⚠️ ОБНАРУЖЕН ДРЕЙФ\nРаспределения статистически различны\n→ Требуется переобучение модели"
    axes[1].text(0.98, 0.15, conclusion, transform=axes[1].transAxes,
                ha='right', va='center', fontsize=11, fontweight='bold', color='red',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='red', linewidth=2))
else:
    conclusion = "✓ ДРЕЙФА НЕТ\nРаспределения статистически схожи\n→ Модель работает стабильно"
    axes[1].text(0.98, 0.15, conclusion, transform=axes[1].transAxes,
                ha='right', va='center', fontsize=11, fontweight='bold', color='green',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='green', linewidth=2))

plt.tight_layout()
plt.show()

print(f"KS-статистика: {ks_stat:.3f}")
print(f"p-value: {p_value:.2e}")
print(f"Вывод: {'ОБНАРУЖЕН ДРЕЙФ ДАННЫХ' if p_value < 0.05 else 'ДРЕЙФА НЕТ'}")

# Итоговое резюме

## Сводная таблица: Статистика → Machine Learning

| Статистический инструмент | Задача в Machine Learning | Когда применять |
| :--- | :--- | :--- |
| **Доверительный интервал (бутстрап)** | Оценка надёжности метрики (Accuracy, F1, AUC) | Всегда, когда публикуете метрику |
| **Проверка гипотез (бутстрап разностей)** | Сравнение двух моделей: "B лучше A или это шум?" | При выборе между алгоритмами |
| **Точный тест Фишера** | Анализ матрицы ошибок на малых данных | N < 30-40 объектов |
| **Хи-квадрат тест** | Отбор категориальных признаков (Feature Selection) | N > 30-40 объектов |
| **KS-тест** | Оценка качества классификации + Data Drift Detection | Сравнение распределений |

## Ключевые выводы

1. Нельзя публиковать метрику без доверительного интервала, потому что точечная оценка может ввести в заблуждение
2. Статистическая значимость ≠ практическая значимость (но без статистической значимости практическая не имеет смысла)
3. KS-тест - идеальный инструмент для нахождения оптимального порога классификации и детекции дрейфа данных
4. Отбор признаком (Feature Selection) можно проводить через классический хи-квадрат

---

## Домашнее задание

Возьмите любой датасет из sklearn.datasets. Обучите две разные модели (например, Logistic Regression и Random Forest). Постройте 95% доверительный интервал для Accuracy каждой модели через бутстрап. Проверьте, статистически ли значима разница между моделями. Выберите 3 категориальных признака и проверьте их значимость через хи-квадрат тест. Постройте KS-тест для предсказанных вероятностей и найдите оптимальный порог классификации.